In [2]:
# 05b – BERT Base (uncased) Inference using model from KaggleHub

import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from tqdm.auto import tqdm
import kagglehub

# -------------------------
# 0. Config
# -------------------------
LABELS = ["anger", "fear", "joy", "sadness", "surprise"]

MODEL_NAME = "bert-base-uncased"
MODEL_HANDLE = "sharmilamamani/emotion-bert/pytorch/bert-base-uncased-v2"
MODEL_FILENAME = "bert_base_uncased_best.pt"

MAX_LEN = 80
THRESHOLD = 0.5
BATCH_SIZE = 16

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -------------------------
# 1. Download model from KaggleHub
# -------------------------
print("Downloading BERT model from KaggleHub...")

model_dir = kagglehub.model_download(MODEL_HANDLE)
model_path = os.path.join(model_dir, MODEL_FILENAME)

print("Model directory:", model_dir)
print("Model path:", model_path)

# -------------------------
# 2. Load test data
# -------------------------
test_df = pd.read_csv("/kaggle/input/2025-sep-dl-gen-ai-project/test.csv")
print("Test samples:", len(test_df))

# -------------------------
# 3. Tokenizer & Dataset
# -------------------------
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

class EmotionDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.df = df
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = str(self.df.iloc[idx]["text"])
        encoding = self.tokenizer(
            text,
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        # Remove extra batch dim from tokenizer outputs
        return {key: val.squeeze(0) for key, val in encoding.items()}

test_dataset = EmotionDataset(test_df, tokenizer)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# -------------------------
# 4. Model definition (must match training)
# -------------------------
class BertForMultiLabel(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.bert = BertModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.linear = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.pooler_output
        pooled = self.dropout(pooled)
        logits = self.linear(pooled)
        return logits  # raw logits

model = BertForMultiLabel(MODEL_NAME, len(LABELS)).to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

print("Model loaded from KaggleHub and ready for inference.")

# -------------------------
# 5. Inference on test set
# -------------------------
all_probs = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Running inference"):
        input_ids = batch["input_ids"].to(device)
        attn = batch["attention_mask"].to(device)

        logits = model(input_ids, attn)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)

all_probs = np.vstack(all_probs)  # shape: (num_samples, num_labels)
preds_bin = (all_probs > THRESHOLD).astype(int)

print("Predictions shape:", preds_bin.shape)

# -------------------------
# 6. Create submission
# -------------------------
submission = pd.DataFrame(preds_bin, columns=LABELS)
submission.insert(0, "id", test_df["id"])

submission_filename = "submission.csv"
submission.to_csv(submission_filename, index=False)

print(f"Saved submission file: {submission_filename}")


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Using device: cuda
Model directory: /kaggle/input/emotion-bert/pytorch/bert-base-uncased-v2/1
Model path: /kaggle/input/emotion-bert/pytorch/bert-base-uncased-v2/1/bert_base_uncased_best.pt
Test samples: 1707


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Model loaded from KaggleHub and ready for inference.


Running inference:   0%|          | 0/107 [00:00<?, ?it/s]

Predictions shape: (1707, 5)
Saved submission file: submission.csv
